# Phase 04 — CIC-IDS2017 Data Cleaning & Leakage Prevention

This notebook orchestrates reusable project modules. Raw source files stay read-only. Assessment results are generated from the current input when the cells run; no example output is fabricated or embedded.

## 1. Project and dataset scope

CIC-IDS2017 is a labelled, controlled network-flow benchmark. This workflow creates an analytical copy for later feature engineering. It does not train models, encode categories, scale features, or split the data. See the official dataset page (https://www.unb.ca/cic/datasets/ids-2017.html) and the project data cleaning policy (../docs/data_cleaning.md).

## 2. Environment and imports

Install the project requirements first. PyArrow is the explicit large-data output format; the cleaner does not silently fall back to CSV.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import pyarrow.parquet as pq

from intrusion_intelligence.config import load_config
from intrusion_intelligence.preprocessing.assessment import assess_dataset
from intrusion_intelligence.preprocessing.policy import load_policy, DEFAULT_POLICY_PATH
from intrusion_intelligence.preprocessing.service import clean_dataset

config = load_config()
source_directory = config.dataset.directory
policy_path = config.project_root / DEFAULT_POLICY_PATH
policy = load_policy(policy_path)
# Select a new output root for each run; existing outputs are never overwritten.
output_root = config.paths.processed_data / f'{config.dataset.name}-notebook-run-01'

## 3. Source discovery and loading strategy

List discovered files and sizes without loading their rows. The assessment and cleaner use chunked, source-by-source reads through the existing validated ingestion interfaces.

In [ ]:
source_files = sorted(source_directory.rglob('*.csv'))
source_inventory = pd.DataFrame([
    {'source_file': path.relative_to(source_directory).as_posix(), 'size_bytes': path.stat().st_size}
    for path in source_files
])
source_inventory

## 4. Schema validation and initial quality scan

The full scan records exact counts. Sampled outlier and cardinality diagnostics are labelled as estimates. Incompatible source schemas fail before outputs are written.

In [ ]:
before = assess_dataset(
    config,
    chunk_size=config.ingestion.chunk_size,
    sample_stride=100,
    compute_duplicates=True,
)
before['schema'], before['dataset_overview']['rows']

## 5. Column and data type analysis

Review exact source headers and the assessment's numeric parsing counts. Header normalization is deterministic; normalized collisions receive source-position suffixes and remain traceable in the manifest.

In [ ]:
feature_type_review = pd.DataFrame([
    {'internal_name': name, 'source_header': result['source_header'],
     'source_position': result['source_position'],
     'invalid_numeric': result['counts']['invalid_numeric']}
    for name, result in before['features'].items()
])
feature_type_review.head(20)

## 6. Missing-value analysis

Missing tokens are counted per feature before any transformation. The default policy canonicalizes them to Parquet nulls and does not impute or discard rows with missing predictive features.

In [ ]:
missing_review = pd.DataFrame([
    {'feature': name, **{key: value for key, value in result['counts'].items() if key in ('empty_string', 'explicit_null_token', 'nan_token')}}
    for name, result in before['features'].items()
]).fillna(0)
missing_review.loc[missing_review.drop(columns='feature').sum(axis=1) > 0]

## 7. Infinite values and numeric conversion

Positive/negative infinity are converted to missing values in the processed analytical copy. Failed numeric conversions become null only after their counts and example tokens are captured. Finite negatives and outliers remain unchanged.

In [ ]:
infinity_review = pd.DataFrame([
    {'feature': name, 'positive_infinity': result['counts']['positive_infinity'],
     'negative_infinity': result['counts']['negative_infinity'],
     'invalid_numeric': result['counts']['invalid_numeric'],
     'examples': result.get('invalid_numeric_examples', [])}
    for name, result in before['features'].items()
])
infinity_review.loc[(infinity_review['positive_infinity'] + infinity_review['negative_infinity'] + infinity_review['invalid_numeric']) > 0]

## 8. Label validation and normalization

Labels are trimmed against an explicit expected-category list. Original label strings are retained in the source-label metadata column. Missing, blank, or unexpected target labels are written to quarantine with a reason; target imputation is never used.

In [ ]:
expected_labels = policy['labels']['expected_values']
label_counts_before = before['dataset_overview']['label_counts']
pd.Series(label_counts_before, name='rows').rename_axis('original_label').to_frame()

## 9. Exact duplicates, feature vectors, and label conflicts

The full assessment uses external fingerprint aggregation. Exact duplicate removal is configurable and disabled by default. Matching feature vectors with conflicting labels are reported and retained for review.

In [ ]:
duplicate_review = before['duplicate_analysis']
duplicate_review

## 10. Identifier, target, feature, and metadata separation

Use the run's feature-policy JSON as the authoritative list. Label, source labels, file/row provenance, record IDs, timestamps, and direct identifiers are excluded from candidate model inputs. Destination port and protocol remain candidates when present. See ../docs/data_leakage_policy.md.

In [ ]:
policy['column_roles'], policy['features'], policy['duplicates']

## 11. Execute the reusable cleaning pipeline

The pipeline rechecks schemas and source checksums, streams chunks, writes typed Parquet, writes a separate quarantine file, and validates outputs before publishing. Existing output paths are not overwritten.

In [ ]:
cleaning_run = clean_dataset(
    config,
    chunk_size=config.ingestion.chunk_size,
    sample_stride=100,
    output_directory=output_root,
    policy_path=policy_path,
)
cleaning_run['status'], cleaning_run['records'], cleaning_run['artifacts']

## 12. Before-and-after comparison

Review row accounting, label distributions, missingness, infinity replacement, conversion failures, and quarantine reasons. Counts come from the current run's generated reports.

In [ ]:
pd.DataFrame({
    'before': pd.Series(cleaning_run['records']['label_counts_before']),
    'after': pd.Series(cleaning_run['records']['label_counts_after']),
}).fillna(0).astype('int64')

## 13. Post-cleaning validation and audit

The cleaner validates Parquet reopenability, unique standardized columns, numeric infinity removal, target validity, quarantine row accounting, source immutability, and feature-policy separation.

In [ ]:
manifest = json.loads(Path(cleaning_run['artifacts']['manifest']).read_text(encoding='utf-8'))
feature_policy = json.loads((output_root / 'metadata' / 'feature_policy.json').read_text(encoding='utf-8'))
label_validation = json.loads((output_root / 'metadata' / 'label_validation.json').read_text(encoding='utf-8'))
validation = cleaning_run['validation']
output_file = config.project_root / manifest['output_files'][0]['path']
validation, feature_policy, label_validation, pq.ParquetFile(output_file).metadata.num_rows, manifest['quarantine']

## 14. Findings and limitations

Use the actual quality and cleaning reports to summarize the run. Negative finite values and outliers are not automatically corrected; exact duplicates are preserved under the default rule; conflicting feature-vector labels remain for investigation. The available extract lacks timestamps, endpoint IPs, standalone source port, protocol, and Flow ID. No split or model was created.